# Vignette 7: Performance by skin tone

The paper's DDI experiment showed synthetic augmentation improving benign-vs-malignant accuracy
across Fitzpatrick skin type groups. DDI needs a data-use agreement, but Fitzpatrick 17k carries
Fitzpatrick scale labels for every image, so the same stratified analysis runs on the sample:
train with and without synthetic images, evaluate on the held-out images grouped by skin type
(I-II, III-IV, V-VI).

In [ ]:
SMALL = True
SYNTHETIC_DIR = None
N_REAL_PER_CLASS = 32
N_SYNTHETIC_PER_REAL = 10
SEEDS = [0] if SMALL else [0, 1, 2]
MAX_EPOCHS = 8 if SMALL else 50

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from synderm.data import load_sample, load_2024_shards, SyntheticDataset
from synderm.data.atlas import HELD_OUT_CSV
real_train, real_val = load_sample(split="train"), load_sample(split="val")
classes = real_train.labels
# Synthetic images derived from validation or held-out real images are excluded (no leakage).
banned = set(real_val.df["md5hash"]) | set(pd.read_csv(HELD_OUT_CSV)["md5hash"])
POLICY = "ref-in-train" if SYNTHETIC_DIR else "ref-not-in-val"
synthetic = SyntheticDataset(SYNTHETIC_DIR, labels=classes) if SYNTHETIC_DIR else load_2024_shards(
    classes, method="finetune-text-to-image", submethod="text-to-image", max_per_label=N_SYNTHETIC_PER_REAL * 64,
    exclude_ref_ids=banned)

In [ ]:
from synderm.utils import synthetic_train_val_split
from synderm.classify import ClassificationDataset, build_transforms, train_classifier, accuracy_ci
GROUPS = {"I-II": {1, 2}, "III-IV": {3, 4}, "V-VI": {5, 6}}
records = []
for with_syn in (False, True):
    for seed in SEEDS:
        train, val = synthetic_train_val_split(real_train, synthetic if with_syn else None, per_class_test_size=8,
                                               n_real_per_class=N_REAL_PER_CLASS, n_synthetic_per_real=N_SYNTHETIC_PER_REAL, random_state=seed, synthetic_policy=POLICY, exclude_ref_ids=banned)
        res = train_classifier(ClassificationDataset(train, classes, build_transforms(augment=True)), ClassificationDataset(val, classes),
                               classes, test_dataset=ClassificationDataset(real_val, classes, group_key="fitzpatrick_scale"),
                               max_epochs=MAX_EPOCHS, seed=seed, verbose=False)
        m = res.test_metrics
        for g, members in GROUPS.items():
            mask = np.isin(np.array(m["groups"]), list(members))
            records.append({"synthetic": with_syn, "seed": seed, "group": g, "n": int(mask.sum()),
                            "correct": int((m["preds"][mask] == m["targets"][mask]).sum())})
        print(f"synthetic={with_syn} seed={seed} overall acc {m['accuracy']:.3f}")
df = pd.DataFrame(records)
agg = df.groupby(["group", "synthetic"]).agg(correct=("correct", "sum"), n=("n", "sum")).reset_index()
agg[["acc", "lo", "hi"]] = agg.apply(lambda r: pd.Series(accuracy_ci(int(r.correct), int(r.n))), axis=1)
agg.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))
w = 0.35
for k, (with_syn, grp) in enumerate(agg.groupby("synthetic")):
    xs = np.arange(len(GROUPS)) + (k - 0.5) * w
    ax.bar(xs, grp.acc, w, yerr=[grp.acc - grp.lo, grp.hi - grp.acc], capsize=3, label="real + synthetic" if with_syn else "real only")
ax.set_xticks(range(len(GROUPS))); ax.set_xticklabels([f"FST {g}\n(n={int(n)})" for g, n in zip(GROUPS, agg[agg.synthetic == False].n)])
ax.set_ylabel("held-out accuracy"); ax.legend(); plt.tight_layout(); plt.show()

Images with an unknown Fitzpatrick scale (-1 in the CSV) are left out of the groups. The sample
is skewed toward types III and IV, so the V-VI bars rest on few images; the paper's DDI
analysis used a skin-type-balanced test set, which is the right design for a full-scale run.